In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

from sklearn.metrics import confusion_matrix

In [2]:
df = pd.read_csv("urinalysis_tests.csv")

print(df.shape)

df.head()

(1436, 16)


,Unnamed: 0,Age,Gender,Color,Transparency,Glucose,Protein,pH,Specific Gravity,WBC,RBC,Epithelial Cells,Mucous Threads,Amorphous Urates,Bacteria,Diagnosis
0,0,76.0,FEMALE,LIGHT YELLOW,CLEAR,NEGATIVE,NEGATIVE,5.0,1.010,1-3,0-2,OCCASIONAL,RARE,NONE SEEN,OCCASIONAL,NEGATIVE
1,1,9.0,MALE,DARK YELLOW,SLIGHTLY HAZY,NEGATIVE,1+,5.0,1.030,1-3,0-2,RARE,FEW,FEW,MODERATE,NEGATIVE
2,2,12.0,MALE,LIGHT YELLOW,SLIGHTLY HAZY,NEGATIVE,TRACE,5.0,1.030,0-3,0-2,RARE,FEW,MODERATE,RARE,NEGATIVE
3,3,77.0,MALE,BROWN,CLOUDY,NEGATIVE,1+,6.0,1.020,5-8,LOADED,RARE,RARE,NONE SEEN,FEW,NEGATIVE
4,4,29.0,FEMALE,YELLOW,HAZY,NEGATIVE,TRACE,6.0,1.025,1-4,0-2,RARE,RARE,NONE SEEN,FEW,NEGATIVE


In [3]:
print(df.isnull().sum())

Unnamed: 0          0
Age                 0
Gender              0
Color               1
Transparency        0
Glucose             0
Protein             0
pH                  0
Specific Gravity    0
WBC                 0
RBC                 0
Epithelial Cells    0
Mucous Threads      0
Amorphous Urates    0
Bacteria            0
Diagnosis           0
dtype: int64


In [4]:
df = df.fillna("UNKNOWN")
print(df.isnull().sum())

Unnamed: 0          0
Age                 0
Gender              0
Color               0
Transparency        0
Glucose             0
Protein             0
pH                  0
Specific Gravity    0
WBC                 0
RBC                 0
Epithelial Cells    0
Mucous Threads      0
Amorphous Urates    0
Bacteria            0
Diagnosis           0
dtype: int64


In [5]:
print(df["Diagnosis"].value_counts())

Diagnosis
NEGATIVE    1355
POSITIVE      81
Name: count, dtype: int64


In [6]:
print(df["Protein"].value_counts())
protein_map = {
    "NEGATIVE": 0,
    "TRACE": 0.5,
    "1+": 1,
    "2+": 2,
    "3+": 3,
    "4+": 4
}

df["Protein"] = df["Protein"].map(protein_map)

Protein
NEGATIVE    804
TRACE       492
1+           94
2+           41
3+            5
Name: count, dtype: int64


In [7]:
print(df["Glucose"].value_counts())
glucose_map = {
    "NEGATIVE": 0,
    "TRACE": 0.5,
    "1+": 1,
    "2+": 2,
    "3+": 3,
    "4+": 4
}

df["Glucose"] = df["Glucose"].map(glucose_map)

Glucose
NEGATIVE    1349
2+            24
3+            23
1+            15
TRACE         13
4+            12
Name: count, dtype: int64


In [8]:
print(df["Bacteria"].value_counts())
bacteria_map = {
    "NONE SEEN":0,
    "RARE":1,
    "FEW":2,
    "OCCASIONAL":3,
    "MODERATE":4,
    "PLENTY":5,
    "LOADED":6
}

df["Bacteria"] = df["Bacteria"].map(bacteria_map)

Bacteria
RARE          755
FEW           434
MODERATE      158
PLENTY         77
OCCASIONAL      8
LOADED          4
Name: count, dtype: int64


In [9]:
print(df["Epithelial Cells"].value_counts())
cell_map = {
    "NONE SEEN":0,
    "RARE":1,
    "FEW":2,
    "OCCASIONAL":3,
    "MODERATE":4,
    "PLENTY":5,
    "LOADED":6
}

df["Epithelial Cells"] = df["Epithelial Cells"].map(cell_map)

Epithelial Cells
RARE          742
FEW           347
MODERATE      188
PLENTY        121
OCCASIONAL     19
NONE SEEN      16
LOADED          3
Name: count, dtype: int64


In [10]:
def convert_range(value):
    value = str(value).strip()
    if ">" in value:
        return float(value.replace(">", ""))
    if "-" in value:
        low, high = value.split("-")
        return (float(low) + float(high)) / 2
    try:
        return float(value)
    except:
        return None

In [11]:
df["WBC"] = df["WBC"].apply(convert_range)
df["RBC"] = df["RBC"].apply(convert_range)

In [12]:
from sklearn.preprocessing import LabelEncoder
categorical_cols =['Gender','Color','Transparency','Amorphous Urates','Mucous Threads']
encoders={}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

In [13]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1436 entries, 0 to 1435
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        1436 non-null   int64  
 1   Age               1436 non-null   float64
 2   Gender            1436 non-null   int64  
 3   Color             1436 non-null   int64  
 4   Transparency      1436 non-null   int64  
 5   Glucose           1436 non-null   float64
 6   Protein           1436 non-null   float64
 7   pH                1436 non-null   float64
 8   Specific Gravity  1436 non-null   float64
 9   WBC               1410 non-null   float64
 10  RBC               1431 non-null   float64
 11  Epithelial Cells  1436 non-null   int64  
 12  Mucous Threads    1436 non-null   int64  
 13  Amorphous Urates  1436 non-null   int64  
 14  Bacteria          1436 non-null   int64  
 15  Diagnosis         1436 non-null   object 
dtypes: float64(7), int64(8), object(1)
memory 

In [15]:
df['Diagnosis'].value_counts()

Diagnosis
NEGATIVE    1355
POSITIVE      81
Name: count, dtype: int64

In [16]:
df['Diagnosis']= df['Diagnosis'].map({'POSITIVE': 1, 'NEGATIVE': 0})

In [17]:
normal_df = df[df["Diagnosis"] == 0].copy()
print(normal_df.shape)

(1355, 16)


Create Features

In [18]:
X_normal = normal_df.drop("Diagnosis",axis=1)

### Standardize Features

In [19]:
scaler = StandardScaler()
X_normal_scaled = scaler.fit_transform(X_normal)

In [20]:
joblib.dump(scaler,"urinalysis_anomaly_scaler.pkl")

['urinalysis_anomaly_scaler.pkl']

### Train Isolation Forest